# Speech to text : Using Python scripts

In [ ]:
!pip install oci

In [1]:
import importlib.metadata

# List the distribution package names
packages = ["oci"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


oci version: 2.180.0


In [2]:
import os, time, json, oci

# --- CONFIGURATION SETTINGS ---
# To get COMPARTMENT_ID -> Identity & Security -> Compartments -> AI-Training
COMPARTMENT_ID = "ocid1.compartment.oc1..aaaaaaaavfpqldn4wptmrkqz5rm5tssleolvw5ns5rc26kuucrun63nxumcq"

# To get bucket: Storage -> Bucket
INPUT_BUCKET = "bucket-speech"
OUTPUT_BUCKET = "bucket-speech"
NAMESPACE = "axzfsaelvve9"             # Found under OCI Profile -> Tenancy
LOCAL_AUDIO_PATH = "sample_audio.mp3"  # Path to your local file
LOCAL_OUTPUT_PATH = "transcription_output.json"         # Where to save the result

# Initialize OCI Clients using default local config profile
config = oci.config.from_file()
object_storage_client = oci.object_storage.ObjectStorageClient(config)
speech_client = oci.ai_speech.AIServiceSpeechClient(config)

print("Success")

Success


# Step 1: Upload the local audio file to the input bucket

In [3]:
def upload_audio():
    """Step 1: Upload the local audio file to the input bucket."""
    filename = os.path.basename(LOCAL_AUDIO_PATH)
    print(f"Uploading {filename} to bucket '{INPUT_BUCKET}'...")
    
    with open(LOCAL_AUDIO_PATH, "rb") as f:
        object_storage_client.put_object(
            namespace_name=NAMESPACE,
            bucket_name=INPUT_BUCKET,
            object_name=filename,
            put_object_body=f
        )
    print("Upload complete.")
    return filename

uploaded_file = upload_audio()

Uploading sample_audio.mp3 to bucket 'bucket-speech'...
Upload complete.


# Step 2: Submit the transcription job to OCI Speech.

In [4]:
def trigger_speech_job(audio_filename):
    """Step 2: Submit the transcription job to OCI Speech."""
    print("Creating OCI Speech Job...")
    
    # Define the input object
    object_location = oci.ai_speech.models.ObjectLocation(
        bucket_name=INPUT_BUCKET,
        namespace_name=NAMESPACE,
        object_names=[audio_filename]
    )
    
    # CORRECTION: Use the correct class for explicit list mappings
    object_list_input_location = oci.ai_speech.models.ObjectListInlineInputLocation(
        location_type="OBJECT_LIST_INLINE_INPUT_LOCATION",
        object_locations=[object_location]
    )
    
    # Define the output destination
    output_location = oci.ai_speech.models.OutputLocation(
        bucket_name=OUTPUT_BUCKET,
        namespace_name=NAMESPACE
    )
    
    # Configure the job details
    model_details = oci.ai_speech.models.TranscriptionModelDetails(
        domain="GENERIC",
        language_code="en-US"
    )
    
    job_details = oci.ai_speech.models.CreateTranscriptionJobDetails(
        compartment_id=COMPARTMENT_ID,
        display_name="Python_Automated_Transcription",
        input_location=object_list_input_location,
        output_location=output_location,
        model_details=model_details
    )
    
    # Submit job
    response = speech_client.create_transcription_job(job_details)
    job_id = response.data.id
    print(f"Job successfully created. Job ID: {job_id}")
    return job_id


speech_job_id = trigger_speech_job(uploaded_file)

Creating OCI Speech Job...
Job successfully created. Job ID: ocid1.aispeechtranscriptionjob.oc1.ap-hyderabad-1.amaaaaaapnafuvaahiyl4zvmxgg2vr54llpfx2oc67oz2nyejhiq3lhykoaq


# Step 3: Poll OCI until the transcription job succeeds.

In [5]:
def wait_for_job(job_id):
    """Step 3: Poll OCI until the transcription job succeeds."""
    print("Waiting for transcription job to process...")
    while True:
        job_status = speech_client.get_transcription_job(job_id).data.lifecycle_state
        print(f"Current Status: {job_status}")
        
        if job_status == "SUCCEEDED":
            print("Job finished successfully!")
            break
        elif job_status in ["FAILED", "CANCELED"]:
            raise Exception(f"Transcription job failed with status: {job_status}")
            
        time.sleep(15)  # Check every 15 seconds to minimize API spam
        
wait_for_job(speech_job_id)


Waiting for transcription job to process...
Current Status: SUCCEEDED
Job finished successfully!


# Step 4: Explicitly construct the file path based on OCI's naming format.

In [6]:
def download_transcription(audio_filename, job_id):
    """Step 4: Explicitly construct the file path based on OCI's naming format."""
    print("Constructing explicit transcription object path...")
    
    # Extract the unique shorthand identifier from the end of the full OCID
    # e.g., 'ocid1.aispeechtranscriptionjob.oc1...amaaaaaapnafu...' -> 'amaaaaaapnafu...'
    job_unique_id = job_id.split('.')[-1]
    
    # Reconstruct the exact path structure you see in your console
    expected_object_name = f"job-{job_unique_id}/{NAMESPACE}_{INPUT_BUCKET}_{audio_filename}.json"
    
    print(f"Downloading explicit target artifact: {expected_object_name}")
    
    try:
        # Fetch the object from the output bucket directly without searching
        get_object_response = object_storage_client.get_object(
            namespace_name=NAMESPACE,
            bucket_name=OUTPUT_BUCKET,
            object_name=expected_object_name
        )
        
        # Save the file locally
        with open(LOCAL_OUTPUT_PATH, "wb") as f:
            for chunk in get_object_response.data.raw.stream(2048, decode_content=False):
                f.write(chunk)
                
        print(f"Successfully downloaded JSON transcription to: {LOCAL_OUTPUT_PATH}")
        
    except oci.exceptions.ServiceError as e:
        if e.status == 404:
            print("⚠️ Direct path failed. Giving OCI 5 seconds to finish writing and trying fallback scan...")
            import time
            time.sleep(5)
            
            # Fallback scanning if it's a structural or timing edge case
            list_response = object_storage_client.list_objects(namespace_name=NAMESPACE, bucket_name=OUTPUT_BUCKET)
            for obj in list_response.data.objects:
                if job_unique_id in obj.name and obj.name.endswith(".json"):
                    print(f"Found artifact via fallback scan: {obj.name}")
                    get_obj = object_storage_client.get_object(NAMESPACE, OUTPUT_BUCKET, obj.name)
                    with open(LOCAL_OUTPUT_PATH, "wb") as f:
                        for chunk in get_obj.data.raw.stream(2048, decode_content=False):
                            f.write(chunk)
                    print(f"Successfully downloaded JSON via fallback to: {LOCAL_OUTPUT_PATH}")
                    return
            raise Exception("File could not be found anywhere in the bucket.")
        else:
            raise e

download_transcription(uploaded_file, speech_job_id)
print("\nPipeline complete! 🚀")


Constructing explicit transcription object path...
Successfully downloaded JSON transcription to: transcription_output.json

Pipeline complete! 🚀


# Print out the json file

In [7]:

import json

with open(LOCAL_OUTPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)
    
# indent=4 formats the JSON with clean spacing and line breaks
print(json.dumps(data, indent=4))

{
    "status": "SUCCESS",
    "timeCreated": "2026-06-24 20:12:46.045",
    "modelDetails": {
        "domain": "GENERIC",
        "languageCode": "en-US",
        "modelType": "ORACLE"
    },
    "audioFormatDetails": {
        "format": "MP3",
        "numberOfChannels": 1,
        "encoding": "MPEG",
        "sampleRateInHz": 48000
    },
    "transcriptions": [
        {
            "transcription": "Hello, students, welcome to the python class. In this session, we are going to be learning about python and its application in artificial intelligence.",
            "confidence": "0.9689",
            "tokens": [
                {
                    "token": "Hello",
                    "startTime": "0.240s",
                    "endTime": "0.480s",
                    "confidence": "0.9746",
                    "type": "WORD"
                },
                {
                    "token": ",",
                    "startTime": "0.480s",
                    "endTime": "0.480s",
   

# print the text

In [8]:
import json

def print_clean_transcription(json_file_path=LOCAL_OUTPUT_PATH):
    """Reads the OCI Speech JSON file and prints the full text transcription directly."""
    print("\n--- 📝 CLEAN TRANSCRIPTION OUTPUT ---")
    
    try:
        with open(json_file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        # Access the list of transcriptions
        transcriptions = data.get("transcriptions", [])
        
        if transcriptions and isinstance(transcriptions, list):
            # Extract the compiled paragraph string directly from the first element
            full_text = transcriptions[0].get("transcription", "")
            confidence = transcriptions[0].get("confidence", "N/A")
            
            print(f"👉 {full_text}")
            print(f"\n📊 Model Confidence: {float(confidence) * 100:.2f}%")
        else:
            print("⚠️ No transcriptions found in the expected format inside this JSON file.")
            
    except FileNotFoundError:
        print(f"❌ Error: Could not find the file '{json_file_path}'.")
    except json.JSONDecodeError:
        print("❌ Error: Failed to parse JSON. Check the file content.")


print_clean_transcription()



--- 📝 CLEAN TRANSCRIPTION OUTPUT ---
👉 Hello, students, welcome to the python class. In this session, we are going to be learning about python and its application in artificial intelligence.

📊 Model Confidence: 96.89%


# print individual words and confidence

In [9]:
import json

def print_word_level_details(json_file_path=LOCAL_OUTPUT_PATH):
    """Reads OCI Speech JSON and extracts word-by-word token metadata."""
    print("\n--- 🔍 DETAILED TOKEN ANALYSIS ---")
    print(f"{'TYPE':<15} | {'TOKEN':<15} | {'CONFIDENCE':<10}")
    print("-" * 48)
    
    try:
        with open(json_file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        transcriptions = data.get("transcriptions", [])
        
        if transcriptions and isinstance(transcriptions, list):
            # Navigate into the tokens array of the first transcription item
            tokens = transcriptions[0].get("tokens", [])
            
            for item in tokens:
                token_text = item.get("token", "")
                token_type = item.get("type", "UNKNOWN")
                confidence = item.get("confidence", "0.0")
                
                # Format confidence as a clean percentage string
                conf_percentage = f"{float(confidence) * 100:.2f}%"
                
                # Print nicely aligned columns for your tutorial output
                print(f"{token_type:<15} | {token_text:<15} | {conf_percentage:<10}")
        else:
            print("⚠️ No tokens found in the expected format.")
            
    except FileNotFoundError:
        print(f"❌ Error: Could not find the file '{json_file_path}'.")
    except ValueError:
        print("❌ Error: Invalid confidence numeric data found in token.")


print_word_level_details()



--- 🔍 DETAILED TOKEN ANALYSIS ---
TYPE            | TOKEN           | CONFIDENCE
------------------------------------------------
WORD            | Hello           | 97.46%    
PUNCTUATION     | ,               | 70.24%    
WORD            | students        | 96.96%    
PUNCTUATION     | ,               | 55.71%    
WORD            | welcome         | 97.10%    
WORD            | to              | 97.07%    
WORD            | the             | 96.53%    
WORD            | python          | 95.92%    
WORD            | class           | 96.60%    
PUNCTUATION     | .               | 98.17%    
WORD            | In              | 97.10%    
WORD            | this            | 97.21%    
WORD            | session         | 96.96%    
PUNCTUATION     | ,               | 76.47%    
WORD            | we              | 97.00%    
WORD            | are             | 96.13%    
WORD            | going           | 96.82%    
WORD            | to              | 96.63%    
WORD            | be   